# Bitcoin Sentiment × Hyperliquid Trader Analysis
### Exploring the intersection of Bitcoin Fear/Greed and Trader Behavior

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Preparation
Loading sentiment data and trade data, and aligning them by date.

In [ ]:
fg = pd.read_csv("fear_greed_index.csv")
fg['date'] = pd.to_datetime(fg['date'])
fg = fg.rename(columns={'classification': 'sentiment', 'value': 'fg_value'})
fg['sentiment_binary'] = fg['sentiment'].map({
    'Extreme Fear': 'Fear', 'Fear': 'Fear',
    'Neutral': 'Neutral',
    'Greed': 'Greed', 'Extreme Greed': 'Greed'
})

hd = pd.read_csv("historical_data.csv")
hd['datetime'] = pd.to_datetime(hd['Timestamp IST'], format='%d-%m-%Y %H:%M')
hd['date'] = pd.to_datetime(hd['datetime'].dt.date)

hd = hd.merge(fg[['date', 'fg_value', 'sentiment', 'sentiment_binary']], on='date', how='left')

## 2. Key Metrics & Segmentation
Aggregating data and creating trader archetypes.

In [ ]:
# Filter to closing events
closing = hd[hd['Direction'].str.contains('Close|Buy|Sell', case=False, na=False)].copy()

trader_stats = closing.groupby('Account').agg(
    total_pnl=('Closed PnL', 'sum'),
    win_rate=('Closed PnL', lambda x: (x > 0).mean()),
    trade_count=('Closed PnL', 'count'),
    avg_size=('Size USD', 'mean')
).reset_index()

# Segmentation
trader_stats['profit_segment'] = np.where(trader_stats['total_pnl'] > 0, 'Winner', 'Loser')
print(trader_stats['profit_segment'].value_counts())

## 3. Performance Analysis
Comparing returns across Fear vs Greed days.

In [ ]:
dt_fg = hd.groupby(['date', 'sentiment_binary']).agg(total_pnl=('Closed PnL', 'sum')).reset_index()
dt_fg = dt_fg[dt_fg['sentiment_binary'].isin(['Fear', 'Greed'])]

sns.boxplot(data=dt_fg, x='sentiment_binary', y='total_pnl', palette={'Fear': '#e74c3c', 'Greed': '#2ecc71'})
plt.title('Daily PnL Distribution: Fear vs Greed')
plt.show()

## Conclusion
Fear days show higher volatility but surprisingly better average performance in this dataset.